# DPR-CTL (2020)

---
[[paper]](https://arxiv.org/pdf/2012.01639)<br>
DPR-CTL = Dense Passage Retrieval with Contrastive Triple Loss

DPR-CTL – это подход, предложенный командой Google Research, который улучшает процесс обучения моделей Dense Retrieval, таких как DPR, за счет использования новой функции потерь и метода генерации негативных примеров.

### Контекст

Задача Dense Passage Retrieval (DPR) заключается в эффективном поиске релевантных текстовых фрагментов (пассажей) по запросу. Модели DPR (2020) показали значительное превосходство над традиционными Sparse Retrieval методами (например, BM25) благодаря использованию Dense Embeddings, полученных от мощных Transformer-энкодеров. Однако обучение этих моделей требует тщательно подобранных, часто человечески аннотированных, негативных примеров для Contrastive Learning, что является ресурсоемкой задачей и ограничивает их применение в сценариях с ограниченным объемом размеченных данных.

### Идея

Основная идея DPR-CTL состоит в том, чтобы **эффективно использовать неразмеченные данные** для обучения моделей Dense Retrieval. Для этого предлагается:
1.  Ввести новую функцию потерь — **Contrastive Triple Loss (CTL)**, которая оперирует тройками (запрос, позитивный пассаж, *синтетический негативный пассаж*).
2.  Разработать метод **автоматической генерации сложных (hard) негативных примеров** из неразмеченных данных, используя претренированные Sequence-to-Sequence модели (например, T5). Эти негативы не являются ни случайными, ни "настоящими" нерелевантными документами, а создаются так, чтобы быть *семантически близкими*, но *фактически нерелевантными* позитивному пассажу.

### Постановка задачи

Решается задача Dense Passage Retrieval: для заданного запроса $Q$ необходимо найти $k$ наиболее релевантных пассажей из базы документов $D = \{d_1, d_2, \ldots, d_N\}$. Релевантность определяется близостью их векторных представлений (embeddings) в латентном пространстве.

### Альтернативные методы

На момент появления DPR-CTL существовали следующие методы, решающие или касающиеся аналогичных задач:
*   **Sparse Retrieval (например, BM25)**: Традиционные методы, основанные на подсчете частоты слов и инвертированной частоте документов. Эффективны, но не улавливают семантическую близость.
*   **DSSM (2013)**: Одна из ранних двухбашенных моделей, использующих нейронные сети для кодирования запросов и документов в плотные векторы, но с ограниченной способностью к пониманию контекста из-за использования поверхностных представлений.
*   **DPR (2020)**: Двухбашенная модель, использующая два BERT-энкодера (один для запроса, другой для пассажа) и Contrastive Learning с in-batch negatives и hard negatives, найденными с помощью BM25. DPR-CTL является прямым улучшением метода обучения DPR.
*   **ANCE (Approximate Nearest Neighbor Negative Contrastive Estimation) (2020)**: Еще один метод улучшения DPR, который фокусируется на динамическом поиске наиболее сложных негативных примеров с использованием ANN-индекса, который обновляется по мере обучения модели. В отличие от ANCE, DPR-CTL больше сфокусирован на синтетических негативах и более эффективном использовании неразмеченных данных.

### Архитектура модели

Архитектура DPR-CTL идентична архитектуре DPR:
*   Это **двухбашенная модель (two-tower model)**, состоящая из двух отдельных, но одинаковых по структуре BERT-энкодеров: **Question Encoder** для запросов и **Passage Encoder** для пассажей.
*   Каждый энкодер принимает текстовый вход и выдает векторное представление (embedding) токена `[CLS]`.
*   Оценка релевантности между запросом $Q$ и пассажем $D$ вычисляется как **скалярное произведение (dot-product)** их соответствующих эмбеддингов: $score(Q, D) = E_Q(Q) \cdot E_D(D)$.

Основное отличие DPR-CTL не в архитектуре, а в *алгоритме обучения*, особенно в используемой функции потерь и подготовке тренировочных данных.

### Алгоритм обучения

Обучение DPR-CTL строится на Contrastive Learning, но с ключевыми модификациями:
1.  **Подготовка данных:**
    *   Для каждого позитивного пассажа $D^+$ и соответствующего запроса $Q^+$ (которые могут быть извлечены из Question Answering датасетов или других источников) генерируется **синтетический негативный пассаж $D^{synth}$**.
    *   Генерация $D^{synth}$ происходит с помощью претренированной Sequence-to-Sequence модели (например, T5). Модели T5 дается $D^+$ и задача сгенерировать парафраз или текст, который *кажется* релевантным, но *не содержит* ответа на запрос или является фактологически неверным/отвлекающим. Например, можно попросить T5 перефразировать $D^+$ так, чтобы он не содержал конкретной сущности, или сгенерировать похожий, но нерелевантный текст. Это делает $D^{synth}$ более сложным негативным примером, чем случайный пассаж.
    *   Таким образом, формируются тройки: $(Q^+, D^+, D^{synth})$.

2.  **Contrastive Triple Loss (CTL):**
    *   Для каждой тройки $(Q^+, D^+, D^{synth})$ вычисляются scores: $s(Q^+, D^+)$ и $s(Q^+, D^{synth})$.
    *   Внутри каждого мини-батча дополнительно используются **in-batch negatives**. Если в батче $N$ пар $(Q_i, D_i^+)$, то для каждого запроса $Q_i$ все другие $D_j^+$ ($j \ne i$) из того же батча рассматриваются как дополнительные негативы.
    *   Функция потерь CTL минимизирует следующую формулу (вариант InfoNCE):
        $$L = -\log \frac{\exp(s(Q, D^+))}{\exp(s(Q, D^+)) + \sum_{D_j \in N_{hard}} \exp(s(Q, D_j))}$$
        Где $N_{hard}$ включает в себя как $D^{synth}$, так и in-batch negatives.
    *   **Ключевое отличие от DPR:** Оригинальный DPR использовал один позитивный пример и один hard negative (часто найденный BM25) для каждого запроса. CTL *добавляет синтетические негативы* и делает упор на генерацию более разнообразных и сложных негативов, что особенно полезно при недостатке вручную аннотированных hard negatives.

3.  **Обучение энкодеров:** Энкодеры обучаются одновременно, чтобы максимизировать скалярное произведение для релевантных пар и минимизировать его для нерелевантных, используя CTL.

### Алгоритм инференса

Алгоритм инференса DPR-CTL идентичен алгоритму DPR:
1.  **Индексация пассажей:** Passage Encoder применяется ко всем пассажам в базе данных. Полученные эмбеддинги сохраняются в ANN-индексе (Approximate Nearest Neighbor), таком как FAISS, для быстрого поиска.
2.  **Кодирование запроса:** Question Encoder применяется к входному запросу $Q$, чтобы получить его эмбеддинг $E_Q(Q)$.
3.  **Поиск:** Эмбеддинг запроса $E_Q(Q)$ используется для поиска $k$ ближайших эмбеддингов пассажей в ANN-индексе.
4.  **Вывод:** Возвращаются соответствующие пассажи как результат поиска.

### Результаты

*   DPR-CTL, будучи обученным на неразмеченных данных с использованием синтетических негативов, демонстрирует значительное улучшение по сравнению с базовой моделью DPR, обученной только на размеченных данных.
*   На датасете Natural Questions (NQ) DPR-CTL превзошел базовый DPR (обученный без жестких негативов из BM25) на 2-3 процентных пункта (п.п.) по точности top-50, используя при этом только синтетические негативы.
*   Наибольший выигрыш DPR-CTL показывает в сценариях, где доступно мало размеченных данных. При обучении только на неразмеченных данных, модель DPR-CTL достигает сопоставимых результатов с полностью обученным DPR, что делает его ценным для сценариев с низкими ресурсами.
*   Интересно, что при комбинировании синтетических негативов с "настоящими" hard negatives (найденными, например, BM25 или ANCE), DPR-CTL показывает дальнейшие улучшения, что указывает на взаимодополняющий характер различных типов негативов.

## 📝 Критический анализ

```markdown
# DPR-CTL (2020)

---
[[paper]](https://arxiv.org/pdf/2012.01639)<br>
DPR-CTL = Dense Passage Retrieval with **Contrastive Triple Loss**

DPR-CTL – это подход от Google Research, улучшающий обучение моделей Dense Retrieval, таких как DPR, с помощью новой функции потерь и метода генерации негативных примеров.

### Контекст

Dense Passage Retrieval (DPR) решает задачу поиска релевантных текстовых фрагментов по запросу. DPR (2020) превосходит Sparse Retrieval методы (например, BM25) благодаря Dense Embeddings, но требует аннотированных негативных примеров для Contrastive Learning, что ограничивает его применение.

### Идея

DPR-CTL эффективно использует неразмеченные данные для обучения моделей Dense Retrieval:
1. Вводит новую функцию потерь — **Contrastive Triple Loss (CTL)**, работающую с тройками (запрос, позитивный пассаж, *синтетический негативный пассаж*).
2. Разрабатывает метод автоматической генерации сложных негативных примеров из неразмеченных данных с помощью Sequence-to-Sequence моделей (например, T5).

### Постановка задачи

Задача Dense Passage Retrieval: для запроса $Q$ найти $k$ наиболее релевантных пассажей из базы документов $D$. Релевантность определяется близостью их векторных представлений.

### Альтернативные методы

- **Sparse Retrieval (BM25)**: Основан на частоте слов, не улавливает семантическую близость.
- **DSSM (2013)**: Двухбашенная модель с ограниченной способностью к пониманию контекста.
- **DPR (2020)**: Использует два BERT-энкодера и Contrastive Learning с in-batch negatives.
- **ANCE (2020)**: Фокусируется на динамическом поиске негативных примеров с использованием ANN-индекса.

### Архитектура

Архитектура DPR-CTL идентична DPR:
- **Двухбашенная модель** с двумя BERT-энкодерами: **Question Encoder** и **Passage Encoder**.
- Релевантность оценивается как **скалярное произведение** эмбеддингов.

### Алгоритм обучения

Обучение DPR-CTL базируется на Contrastive Learning с модификациями:
1. **Подготовка данных**: Генерация синтетических негативных примеров $D^{synth}$ с помощью T5.
2. **Contrastive Triple Loss (CTL)**: Минимизирует потери, используя синтетические и in-batch негативы.
3. **Обучение энкодеров**: Максимизация скалярного произведения для релевантных пар.

### Алгоритм инференса

Идентичен DPR:
1. **Индексация пассажей**: Passage Encoder применяется ко всем пассажам, эмбеддинги сохраняются в ANN-индексе.
2. **Кодирование запроса**: Получение эмбеддинга запроса.
3. **Поиск**: Использование эмбеддинга запроса для поиска ближайших эмбеддингов пассажей.
4. **Вывод**: Возвращаются релевантные пассажи.

### Результаты

- DPR-CTL улучшает точность на 2-3 п.п. по сравнению с базовым DPR на датасете Natural Questions, используя только синтетические негативы.
- Особенно эффективен в сценариях с ограниченными размеченными данными, достигая сопоставимых результатов с полностью обученным DPR.
- Комбинация синтетических и "настоящих" негативов дает дополнительные улучшения.

<img src="img/img.png" width=500>
```


## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
# Импортируем необходимые библиотеки
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch
import numpy as np

# Инициализируем модель и токенайзер T5 для генерации синтетических негативов
tokenizer = T5Tokenizer.from_pretrained('t5-small')
model = T5ForConditionalGeneration.from_pretrained('t5-small')

# Функция для генерации синтетического негативного примера
def generate_synthetic_negative(passage):
    # Формируем вход для модели T5
    input_text = f"paraphrase: {passage} </s>"
    input_ids = tokenizer.encode(input_text, return_tensors="pt")

    # Генерируем выходной текст
    outputs = model.generate(input_ids, max_length=50, num_return_sequences=1)
    synthetic_negative = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    return synthetic_negative

# Пример позитивного пассажа
positive_passage = "The capital of France is Paris."

# Генерируем синтетический негативный пассаж
synthetic_negative_passage = generate_synthetic_negative(positive_passage)
print("Positive Passage:", positive_passage)
print("Synthetic Negative Passage:", synthetic_negative_passage)

# Пример функции потерь Contrastive Triple Loss (CTL)
def contrastive_triple_loss(query_embedding, positive_embedding, synthetic_negative_embedding, in_batch_negatives):
    # Вычисляем скалярные произведения
    pos_score = np.dot(query_embedding, positive_embedding)
    synth_neg_score = np.dot(query_embedding, synthetic_negative_embedding)
    
    # Вычисляем in-batch negative scores
    in_batch_neg_scores = [np.dot(query_embedding, neg) for neg in in_batch_negatives]
    
    # Формируем список всех негативных скорингов
    all_negative_scores = [synth_neg_score] + in_batch_neg_scores
    
    # Вычисляем CTL
    loss = -np.log(np.exp(pos_score) / (np.exp(pos_score) + sum(np.exp(score) for score in all_negative_scores)))
    
    return loss

# Пример эмбеддингов (в реальности они должны быть получены от энкодеров)
query_embedding = np.array([0.1, 0.2, 0.3])
positive_embedding = np.array([0.1, 0.2, 0.3])
synthetic_negative_embedding = np.array([0.3, 0.2, 0.1])
in_batch_negatives = [np.array([0.2, 0.1, 0.3]), np.array([0.3, 0.3, 0.3])]

# Вычисляем функцию потерь
loss = contrastive_triple_loss(query_embedding, positive_embedding, synthetic_negative_embedding, in_batch_negatives)
print("Contrastive Triple Loss:", loss)
```

### Комментарии к коду:

1. **Генерация синтетических негативов:**
   - Используем модель T5 для генерации синтетических негативных примеров. Входной текст формируется как задача перефразирования, чтобы создать текст, который семантически близок, но фактически нерелевантен.

2. **Contrastive Triple Loss (CTL):**
   - CTL минимизирует разницу между скалярным произведением запроса и позитивного пассажа и скалярным произведением запроса и негативных примеров (включая синтетические и in-batch негативы).
   - Используем логарифмическую функцию потерь, чтобы усилить различие между позитивными и негативными примерами.

3. **Эмбеддинги:**
   - В примере используются фиктивные эмбеддинги. В реальном сценарии они должны быть получены от обученных энкодеров модели DPR-CTL.

Этот код иллюстрирует ключевые аспекты метода DPR-CTL, включая генерацию синтетических негативов и использование новой функции потерь CTL.